# 11: Robustness to image processing (E3)

Images on the internet are rarely the original file: they are re-compressed, shrunk and sometimes blurred. Here we take the three trained detectors from E1 (no new training) and test them on processed copies of the cleaned test set:

| Processing | Levels | What it simulates |
|---|---|---|
| JPEG compression | quality 95, 85, 75, 65, 50 | re-saving, uploading to websites and apps |
| Resize | 90%, 75%, 60%, 50% of the size, then back to 224 px | thumbnails, lower-resolution copies |
| Gaussian blur | sigma 0.5, 1.0, 1.5, 2.0 pixels | soft focus, smoothing filters |

Real and generated test images get exactly the same processing, so the comparison stays fair. We also report **M4**, the average of the M2 and M3 scores (no extra training).

GPU runtime needed (about 30 minutes). Results are saved after every setting, so a disconnected run resumes where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import io
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from torchvision.models import resnet50
from transformers import CLIPVisionModelWithProjection

ROOT = Path("/content/drive/MyDrive/minor-project")
LOCAL = Path("/content/processed")
RESULTS = ROOT / "results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
PERTURBATIONS = {
    "JPEG quality": [95, 85, 75, 65, 50],
    "Resize scale": [0.9, 0.75, 0.6, 0.5],
    "Blur sigma": [0.5, 1.0, 1.5, 2.0],
}
CLIP_NAME = "openai/clip-vit-large-patch14"
BATCH_SIZE = 128
OUT = "e3"
COLORS = {"M1 ResNet-50": "#2a78d6", "M2 CLIP": "#eb6834", "M3 NPR": "#1baf7a", "M4 M2+M3": "#eda100"}

## Processing functions

Each function takes one 224 x 224 image (a NumPy array) and returns the processed image at the same size.

In [ ]:
def jpeg(img, quality):
    buffer = io.BytesIO()
    Image.fromarray(img).save(buffer, format="JPEG", quality=int(quality))
    buffer.seek(0)
    return np.asarray(Image.open(buffer).convert("RGB"))


def resize(img, scale):
    size = img.shape[0]
    small = Image.fromarray(img).resize((round(size * scale),) * 2, Image.BILINEAR)
    return np.asarray(small.resize((size, size), Image.BILINEAR))


def blur(img, sigma):
    return np.asarray(Image.fromarray(img).filter(ImageFilter.GaussianBlur(radius=sigma)))


FUNCTIONS = {"JPEG quality": jpeg, "Resize scale": resize, "Blur sigma": blur}


def process_all(images, name, level):
    fn = FUNCTIONS[name]
    return np.stack([fn(np.array(img), level) for img in images])

## Test data

In [ ]:
LOCAL.mkdir(exist_ok=True)
for name in ["test_images.npy", "test_meta.csv"]:
    if not (LOCAL / name).exists():
        shutil.copy(ROOT / "processed" / name, LOCAL / name)

test_images = np.load(LOCAL / "test_images.npy", mmap_mode="r")
test_meta = pd.read_csv(LOCAL / "test_meta.csv")
test_meta["y"] = (test_meta["label"] == "fake").astype(int)
labels = test_meta["y"].to_numpy()
print(f"test: {len(test_meta)} images ({labels.sum()} fake)")

In [ ]:
picture = np.array(test_images[test_meta.index[test_meta["generator"] == "SD15"][0]])
examples = [("Original", picture)] + [
    (f"{name.split()[0]} {levels[-1]}", FUNCTIONS[name](picture, levels[-1])) for name, levels in PERTURBATIONS.items()
]
fig, axes = plt.subplots(2, len(examples), figsize=(3 * len(examples), 6.6))
for col, (title, img) in enumerate(examples):
    axes[0, col].imshow(img)
    axes[0, col].set_title(title, fontsize=10)
    axes[1, col].imshow(img[80:144, 80:144], interpolation="nearest")
    axes[1, col].set_title("64 x 64 zoom", fontsize=9)
    for ax in axes[:, col]:
        ax.axis("off")
plt.tight_layout(h_pad=1.5)
plt.savefig(RESULTS / f"{OUT}_examples.png", dpi=110)
plt.show()

## Load the three trained detectors

M1 and M3 are loaded from their checkpoints. The M2 classifier is re-fitted on the saved CLIP training features with the same C as in notebook 06 (the fit is deterministic), so it is the same classifier.

In [ ]:
class NPRNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet50(weights=None)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
            base.bn1, base.relu, base.maxpool,
        )
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        rebuilt = F.interpolate(F.interpolate(x, scale_factor=0.5, mode="nearest"), scale_factor=2, mode="nearest")
        npr = x - rebuilt
        x = self.stem(npr * 2.0 / 3.0)
        x = self.layer2(self.layer1(x))
        return self.fc(torch.flatten(self.pool(x), 1))


m1 = resnet50(weights=None)
m1.fc = nn.Linear(m1.fc.in_features, 1)
m1.load_state_dict(torch.load(ROOT / "checkpoints" / "m1_resnet50.pt", map_location=device))
m1 = m1.to(device).eval()

m3 = NPRNet()
m3.load_state_dict(torch.load(ROOT / "checkpoints" / "m3_npr.pt", map_location=device))
m3 = m3.to(device).eval()

dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size

best_C = json.load(open(RESULTS / "m2_clip.json"))["best_C"]
train_feats = np.load(ROOT / "features" / f"{CLIP_NAME.split('/')[-1]}_train.npy")
train_y = (pd.read_csv(ROOT / "processed" / "train_meta.csv")["label"] == "fake").astype(int).to_numpy()
clf = LogisticRegression(C=best_C, max_iter=5000).fit(train_feats, train_y)
print(f"Loaded M1, M3 and CLIP; M2 classifier re-fitted with C = {best_C}")

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def score_all(images):
    probs = {"M1 ResNet-50": [], "M2 CLIP": [], "M3 NPR": []}
    for start in range(0, len(images), BATCH_SIZE):
        x = torch.from_numpy(np.array(images[start:start + BATCH_SIZE])).to(device).permute(0, 3, 1, 2).float() / 255
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            imagenet_x = (x - IMAGENET_MEAN) / IMAGENET_STD
            probs["M1 ResNet-50"].append(torch.sigmoid(m1(imagenet_x).squeeze(1).float()).cpu().numpy())
            probs["M3 NPR"].append(torch.sigmoid(m3(imagenet_x).squeeze(1).float()).cpu().numpy())
        clip_x = x if x.shape[-1] == CLIP_SIZE else F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((clip_x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        probs["M2 CLIP"].append(F.normalize(emb, dim=1).cpu().numpy())
    probs = {k: np.concatenate(v) for k, v in probs.items()}
    probs["M2 CLIP"] = clf.predict_proba(probs["M2 CLIP"])[:, 1]
    probs["M4 M2+M3"] = (probs["M2 CLIP"] + probs["M3 NPR"]) / 2
    return probs

## Run every setting (about 2 minutes each)

The unprocessed test set comes first; its numbers should match E1 (M1 0.965, M2 0.975, M3 0.966).

In [ ]:
PROBS_FILE = RESULTS / f"{OUT}_probs.npz"
saved = dict(np.load(PROBS_FILE)) if PROBS_FILE.exists() else {}

settings = [("None", 0)] + [(name, level) for name, levels in PERTURBATIONS.items() for level in levels]
for name, level in settings:
    key = f"{name}={level}"
    if any(k.startswith(key + "|") for k in saved):
        print(f"{key}: already done")
        continue
    images = test_images if name == "None" else process_all(test_images, name, level)
    for model, probs in score_all(images).items():
        saved[f"{key}|{model}"] = probs
    np.savez(PROBS_FILE, **saved)
    accuracy = {m.split()[0]: round(float(((saved[f'{key}|{m}'] >= 0.5) == labels).mean()), 3) for m in COLORS}
    print(f"{key}: accuracy {accuracy}")

## Summary table

In [ ]:
rows = []
for key, probs in saved.items():
    setting, model = key.split("|")
    name, level = setting.split("=")
    predicted_fake = probs >= 0.5
    rows.append({
        "model": model,
        "perturbation": name,
        "level": float(level),
        "accuracy": float((predicted_fake == labels).mean()),
        "auc": float(roc_auc_score(labels, probs)),
        "real_correct": float((~predicted_fake[labels == 0]).mean()),
        "fake_detected": float(predicted_fake[labels == 1].mean()),
    })
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / f"{OUT}_summary.csv", index=False)

for name in PERTURBATIONS:
    part = summary[summary["perturbation"].isin(["None", name])].copy()
    part["level"] = part["level"].astype(str).where(part["perturbation"] != "None", "none")
    order = ["none"] + [str(float(level)) for level in PERTURBATIONS[name]]
    for metric in ["accuracy", "auc"]:
        print(f"\n{name}: {metric}")
        display(part.pivot(index="model", columns="level", values=metric)[order].round(3))

## Charts

In [ ]:
def plot(metric, ylabel, filename):
    fig, axes = plt.subplots(1, len(PERTURBATIONS), figsize=(5 * len(PERTURBATIONS), 4.2), sharey=True)
    for ax, name in zip(axes, PERTURBATIONS):
        ticks = ["none"] + [str(level) for level in PERTURBATIONS[name]]
        for model, color in COLORS.items():
            values = [summary.query("model == @model and perturbation == 'None'")[metric].iloc[0]]
            for level in PERTURBATIONS[name]:
                values.append(summary.query("model == @model and perturbation == @name and level == @level")[metric].iloc[0])
            ax.plot(range(len(ticks)), np.array(values) * 100, marker="o", markersize=6, linewidth=2, color=color, label=model)
        ax.set_xticks(range(len(ticks)), ticks)
        ax.set_xlabel(name)
        ax.axhline(50, color="#999999", linewidth=1, linestyle="--")
        ax.grid(axis="y", color="#e5e5e5")
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel(ylabel)
    axes[0].set_ylim(min(40, np.floor(summary[metric].min() * 10) * 10 - 5), 101)
    handles, names = axes[0].get_legend_handles_labels()
    fig.legend(handles, names, loc="upper center", ncol=len(COLORS), frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.9])
    plt.savefig(RESULTS / filename, dpi=120)
    plt.show()


plot("accuracy", "accuracy (%)", f"{OUT}_accuracy.png")
plot("auc", "ROC-AUC x 100", f"{OUT}_auc.png")